In [ ]:
# === Cell 0: Setup & Imports ===
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.optimizers import Adam

print("TensorFlow version:", tf.__version__)

IMG_SIZE = (380, 380)
BATCH_SIZE = 16
NUM_CLASSES = 26
EPOCHS = 10

TRAIN_DIR = '/kaggle/input/cropcare-mukta-26classes/train/train'
VAL_DIR   = '/kaggle/input/cropcare-mukta-26classes/val/val'
TEST_DIR  = '/kaggle/input/cropcare-mukta-26classes/test/test'

assert os.path.exists(TRAIN_DIR), f"TRAIN_DIR does not exist: {TRAIN_DIR}"
assert os.path.exists(VAL_DIR),   f"VAL_DIR does not exist: {VAL_DIR}"
assert os.path.exists(TEST_DIR),  f"TEST_DIR does not exist: {TEST_DIR}"
print("✔ Dataset directories exist.")


: 

In [ ]:
# === Cell 1: Load datasets via tf.data ===
train_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    label_mode='categorical',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)
val_ds = keras.utils.image_dataset_from_directory(
    VAL_DIR,
    label_mode='categorical',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)
test_ds = keras.utils.image_dataset_from_directory(
    TEST_DIR,
    label_mode='categorical',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Classes:", train_ds.class_names)
print("Number of classes:", len(train_ds.class_names))


In [ ]:
# === Cell 2: Preprocessing + Data Augmentation Pipeline ===
# We'll use built-in tf.keras layers for augmentation — simpler and GPU-friendly. 

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
    # You can optionally add more (brightness, shift, etc.)
], name="data_augmentation")

def preprocess(image, label):
    image = tf.cast(image, tf.float32)
    return keras.applications.efficientnet.preprocess_input(image), label

def augment_and_preprocess(image, label):
    image = data_augmentation(image, training=True)
    return preprocess(image, label)

train_ds = (
    train_ds
    .map(augment_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    val_ds
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    test_ds
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

print("✔ Data pipeline (augmentation + preprocess) ready")


In [ ]:
# === Cell 3: Moderately Advanced Leaf Disease Model ===
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Backbone: EfficientNetB4 (more detailed features than B3)
backbone = keras.applications.EfficientNetB4(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)
backbone.trainable = False  # freeze for first training stage


# ---- Lightweight Attention (NO TF ops, NO errors) ----
def AttentionLite(x):
    # Global average pooling for channel info
    gap = layers.GlobalAveragePooling2D()(x)
    dense1 = layers.Dense(x.shape[-1] // 8, activation="relu")(gap)
    dense2 = layers.Dense(x.shape[-1], activation="sigmoid")(dense1)
    scale = layers.Reshape((1, 1, x.shape[-1]))(dense2)
    return layers.Multiply()([x, scale])


# ---- Model Input ----
inputs = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))

# Backbone feature extraction
x = backbone(inputs, training=False)

# Add lightweight attention
x = AttentionLite(x)

# Global pooling
x = layers.GlobalAveragePooling2D()(x)

# Strong classification head
x = layers.Dropout(0.45)(x)
x = layers.Dense(512, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.35)(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.25)(x)

# Output layer
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

# Build model
model = keras.Model(inputs, outputs, name="LeafDisease_AdvancedLite")

model.summary()


In [ ]:
# === Cell 4: Dataset Loading + Moderate Real-World Augmentation ===
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory

# Use the directories detected earlier
print("✔ Using: ")
print("TRAIN_DIR =", TRAIN_DIR)
print("VAL_DIR   =", VAL_DIR)
print("TEST_DIR  =", TEST_DIR)

# ===== DATASET LOAD =====
train_raw = image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    label_mode="categorical"
)

val_raw = image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode="categorical"
)

test_raw = image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode="categorical"
)

print("✔ Dataset loaded successfully.")


# ===== AUGMENTATION (SAFE VERSION) =====
data_augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.15),
    tf.keras.layers.RandomBrightness(0.10),
], name="augmentation_block")


# ===== PREPROCESSING =====
from tensorflow.keras.applications.efficientnet import preprocess_input

def process_train(images, labels):
    images = data_augment(images)                           # apply aug
    images = preprocess_input(tf.cast(images, tf.float32))  # efficientnet format
    return images, labels

def process_eval(images, labels):
    images = preprocess_input(tf.cast(images, tf.float32))
    return images, labels


# ===== APPLY FINAL PIPELINES =====
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_raw.map(process_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = val_raw.map(process_eval,  num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds  = test_raw.map(process_eval, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

print("✔ Final training-ready dataset created.")
print("🔹 Train batches:", len(train_ds))
print("🔹 Val batches:  ", len(val_ds))
print("🔹 Test batches: ", len(test_ds))


In [ ]:
# === Cell 5: Compile Model + Advanced Callbacks ===

from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    ReduceLROnPlateau,
    EarlyStopping,
    CSVLogger
)

# ===== OPTIMIZER =====
optimizer = AdamW(
    learning_rate=1e-4,
    weight_decay=1e-5
)


# ===== CALLBACKS =====

# 1) Save your best model even if training stops/restarts
checkpoint_cb = ModelCheckpoint(
    "best_cropcare_model.h5",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

# 2) Reduce LR when accuracy stops improving
lr_cb = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

# 3) Stop training if truly stuck
earlystop_cb = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True,
    verbose=1
)

# 4) Log training progress (you can check later if system restarts)
csv_logger = CSVLogger("training_log.csv")


# ===== COMPILE MODEL =====
model.compile(
    optimizer=optimizer,
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("✔ Model compiled successfully.")
print("✔ Callbacks ready.")


In [ ]:
# === Cell 6: Train the Model ===

#EPOCHS = 8

#history = model.fit(
   # train_ds,
   # validation_data=val_ds,
   # epochs=EPOCHS,
    #callbacks=[checkpoint_cb, lr_cb, earlystop_cb, csv_logger],
    #verbose=1)

print("\n🎉 Training completed successfully!")


In [ ]:
model.save("/kaggle/working/final_model.h5")
model.save("/kaggle/working/final_model.keras")


In [ ]:
from tensorflow.keras.models import load_model

model = load_model("/kaggle/input/trained-data/best_cropcare_model.h5", compile=False)

print("✔ Model loaded successfully!")


In [ ]:
import os

class_dir = "/kaggle/input/cropcare-mukta-26classes/train/train"
class_names = sorted(os.listdir(class_dir))

print("Total classes:", len(class_names))
print(class_names)


In [ ]:
import os

folder = "/kaggle/input/cropcare-mukta-26classes/train/train/Apple___healthy"
files = os.listdir(folder)
print("Total images:", len(files))
print(files[:20])   # show first 20 images


In [ ]:
# >>> RUN THIS CELL (after model has been loaded)
# It picks a random image from your train folder, shows it, and prints the model's prediction.

import os, random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# --- 1) paths & class names (already detected earlier) ---
DATA_ROOT = "/kaggle/input/cropcare-mukta-26classes/train/train"   # change only if your path differs
CLASS_NAMES = sorted(os.listdir(DATA_ROOT))
print("Using data root:", DATA_ROOT)
print("Num classes:", len(CLASS_NAMES))

# --- 2) get target input size from loaded model (works if model variable exists) ---
try:
    # model must be previously loaded: model = load_model(...)
    input_shape = model.input_shape  # e.g. (None, 380, 380, 3)
    target_size = (int(input_shape[1]), int(input_shape[2]))
except Exception as e:
    print("Could not read model.input_shape:", e)
    target_size = (224, 224)   # safe fallback
print("Target size used for prediction:", target_size)

# --- 3) choose random class folder and a random image inside it ---
cls = random.choice(CLASS_NAMES)
folder = os.path.join(DATA_ROOT, cls)
img_name = random.choice(os.listdir(folder))
img_path = os.path.join(folder, img_name)

print("Chosen folder (true class):", cls)
print("Chosen image:", img_name)
print("Full path:", img_path)

# --- 4) load image and preprocess ---
# Try using EfficientNet preprocess if available (safer if you trained with EfficientNet);
# otherwise fall back to simple scaling /255.0
def load_and_preprocess(path, size):
    img = tf.keras.utils.load_img(path, target_size=size)
    arr = tf.keras.utils.img_to_array(img)  # float32
    arr = np.expand_dims(arr, axis=0)
    # try EfficientNet preprocess_input, else use /255
    try:
        from tensorflow.keras.applications.efficientnet import preprocess_input
        arr = preprocess_input(arr.astype("float32"))
        used_pre = "efficientnet.preprocess_input"
    except Exception:
        arr = arr.astype("float32") / 255.0
        used_pre = "scaled /255.0"
    return arr, used_pre, img

img_array, used_pre, pil_img = load_and_preprocess(img_path, target_size)
print("Preprocessing used:", used_pre)

# --- 5) prediction ---
preds = model.predict(img_array)
pred_idx = int(np.argmax(preds, axis=1)[0])
pred_class = CLASS_NAMES[pred_idx]
confidence = float(np.max(preds))

print(f"\nModel prediction: {pred_class}  (confidence: {confidence*100:.2f}%)")

# --- 6) show the image and result ---
plt.figure(figsize=(5,5))
plt.imshow(pil_img)
plt.axis("off")
plt.title(f"True: {cls}\nPred: {pred_class} ({confidence*100:.2f}%)")
plt.show()


In [ ]:
import tensorflow as tf
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# -------------------------------
# 1) Load Test Dataset
# -------------------------------
TEST_DIR = "/kaggle/input/cropcare-mukta-26classes/test/test"

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TEST_DIR,
    label_mode="categorical",
    image_size=(target_size[0], target_size[1]),
    shuffle=False
)

class_names = test_ds.class_names
print("Classes:", class_names)

# -------------------------------
# 2) Run Predictions
# -------------------------------
y_true = []
y_pred = []

for batch_images, batch_labels in test_ds:
    preds = model.predict(batch_images)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(np.argmax(batch_labels.numpy(), axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# -------------------------------
# 3) Show Accuracy
# -------------------------------
accuracy = (y_true == y_pred).mean() * 100
print(f"\n✔ FINAL TEST ACCURACY: {accuracy:.2f}%")

# -------------------------------
# 4) Classification Report
# -------------------------------
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

# -------------------------------
# 5) Confusion Matrix
# -------------------------------
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(16,12))
sns.heatmap(cm, annot=False, cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
# Run this after you have y_true, y_pred and class_names from your evaluation
import numpy as np, os, matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)
np.fill_diagonal(cm, 0)   # zero out diagonal to focus on confusions

# top N confusions
N = 10
pairs = []
num_classes = len(class_names)
for i in range(num_classes):
    for j in range(num_classes):
        pairs.append(((i,j), cm[i,j]))
pairs_sorted = sorted(pairs, key=lambda x: x[1], reverse=True)
top_pairs = [p for p in pairs_sorted if p[1] > 0][:N]

print("Top confusion pairs (true -> predicted) and counts:")
for (i,j),c in top_pairs:
    print(f"{class_names[i]}  ->  {class_names[j]}  :  {c}")

# show 3 examples of first top confusion
true_idx, pred_idx = top_pairs[0][0]
root = "/kaggle/input/cropcare-mukta-26classes/test/test"
true_folder = os.path.join(root, class_names[true_idx])
examples = [f for f in os.listdir(true_folder)]
# random pick a few examples — you might want to pick specific misclassified files by repeating evaluation with file paths
for ex in examples[:3]:
    img = plt.imread(os.path.join(true_folder, ex))
    plt.figure(figsize=(4,4)); plt.imshow(img); plt.axis('off'); plt.title(f"True: {class_names[true_idx]} -> Predicted often as: {class_names[pred_idx]}")


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

labels = y_true  # from evaluation or build from dataset folder counts
classes = np.unique(labels)
class_weights = compute_class_weight("balanced", classes=classes, y=labels)
class_weight_dict = {i: w for i,w in zip(classes, class_weights)}
print(class_weight_dict)
# then pass class_weight=class_weight_dict to model.fit(...)


In [ ]:
import seaborn as sns, matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred, normalize='true')  # normalize by true class
plt.figure(figsize=(14,12))
sns.heatmap(cm, vmax=1.0, cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title("Normalized Confusion Matrix (rows=true classes)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# class names from earlier
class_names = CLASS_NAMES  
num_classes = len(class_names)

# get labels from train directory
train_paths = []
train_labels = []

for i, cls in enumerate(class_names):
    folder = f"/kaggle/input/cropcare-mukta-26classes/train/train/{cls}"
    for img in os.listdir(folder):
        train_paths.append(folder + "/" + img)
        train_labels.append(i)

train_labels = np.array(train_labels)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(num_classes),
    y=train_labels
)

class_weights = dict(enumerate(class_weights))
class_weights


In [ ]:
# Unfreeze deeper layers
for layer in model.layers[-40:]:
    layer.trainable = True

from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(1e-5),  # small LR for safety
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("✔ Model ready for fine-tuning.")


In [ ]:
#history_ft = model.fit(
   # train_ds,
   # validation_data=val_ds,
    #epochs=6,
    #class_weight=class_weights,
    #callbacks=[
       # ReduceLROnPlateau(monitor="val_loss", patience=2, factor=0.4, verbose=1),
     #   EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True) ])


In [ ]:
model.save("cropcare_finetuned_best.keras")
print("Model saved in new Keras format!")


In [ ]:
from kaggle_secrets import UserSecretsClient
from IPython.display import FileLinks

FileLinks('/kaggle/working')


In [ ]:
from IPython.display import FileLink
FileLink("cropcare_finetuned_best.keras")
